<a href="https://colab.research.google.com/github/omganeshkn-coder/Applied-probability-statistics-LAB/blob/main/SmartCropRecommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import ee

ee.Authenticate()
ee.Initialize(project="crop-reco-me")

farm = ee.Geometry.Point([74.76, 13.42])   # (longitude, latitude) near Brahmavar

elev = ee.Image("USGS/SRTMGL1_003").sample(farm, 30).first().get("elevation").getInfo()
ph   = ee.Image("OpenLandMap/SOL/SOL_PH-H2O_USDA-4C1A2A_M/v02").select("b0").sample(farm, 250).first().get("b0").getInfo()

print("Elevation (m):", elev)
print("Topsoil pH (map estimate):", ph / 10)

Elevation (m): 31
Topsoil pH (map estimate): 4.9


In [2]:
# Connect Colab to your Google Drive so it can read the file
from google.colab import drive
drive.mount('/content/drive')

import glob
import pandas as pd

# Find the CSV you uploaded
files = glob.glob('/content/drive/MyDrive/crop_reco/raw/*.csv')
print("Files found:", files)

df = pd.read_csv(files[0])
print("\nWhole dataset: rows, columns =", df.shape)
print("Column names:", df.columns.tolist())
print(df.head(3))

# Keep only Karnataka
ka = df[df['state_name'].astype(str).str.strip().str.lower() == 'karnataka']
print("\nKarnataka rows:", len(ka))
print("Years:", sorted(ka['year'].unique()))
print("Number of districts:", ka['district_name'].nunique())
print(sorted(ka['district_name'].unique()))
print("Seasons:", ka['season'].unique())
print("\nTop 25 crops by number of rows:")
print(ka['crop_name'].value_counts().head(25))
print("\nShare of missing values:")
print(ka[['area', 'production', 'yield']].isna().mean().round(3))

Mounted at /content/drive
Files found: ['/content/drive/MyDrive/crop_reco/raw/crop-wise-area-production-yield.csv']

Whole dataset: rows, columns = (455359, 16)
Column names: ['id', 'year', 'state_name', 'state_code', 'district_name', 'district_code', 'crop_name', 'crop_code', 'crop_type', 'season', 'area', 'area_unit', 'production', 'production_unit', 'yield', 'yield_unit']
   id       year      state_name  state_code  district_name  district_code  \
0   0  1997-1998  Andhra Pradesh          28  Ananthapuramu            502   
1   1  1997-1998  Andhra Pradesh          28  Ananthapuramu            502   
2   2  1997-1998  Andhra Pradesh          28  Ananthapuramu            502   

     crop_name  crop_code crop_type  season     area area_unit  production  \
0    Arhar/Tur      202.0    Pulses  Kharif  21400.0   Hectare      2600.0   
1        Bajra      103.0   Cereals  Kharif   1400.0   Hectare       500.0   
2  Castor Seed     1002.0  Oilseeds  Kharif   1000.0   Hectare       100.0 

In [3]:
# 1. Full district list, one per line (the earlier print was cut off)
print("\n".join(sorted(ka['district_name'].unique())))

# 2. Rows per year: shows where the data thins out
print("\nRows per year (last 10):")
print(ka.groupby('year').size().tail(10))

# 3. Missing values and units
print("\nShare missing:")
print(ka[['area', 'production', 'yield']].isna().mean().round(3))
print("\nYield units:", ka['yield_unit'].value_counts().to_dict())
print("Production units:", ka['production_unit'].value_counts().to_dict())
print("\nYield summary (tonnes/hectare):")
print(ka['yield'].describe().round(3))
print("Rows with zero yield:", (ka['yield'] == 0).sum())

# 4. The coast: which crops and seasons have records?
coast_names = [d for d in ka['district_name'].unique()
               if 'udupi' in d.lower() or 'dakshin' in d.lower()]
print("\nCoastal districts found:", coast_names)
coast = ka[ka['district_name'].isin(coast_names)]
print("Coastal rows:", len(coast))
print(coast.groupby(['district_name', 'season', 'crop_name'])
           .size().sort_values(ascending=False).head(40))

Bagalkote
Ballari
Belagavi
Bengaluru Rural
Bengaluru South
Bengaluru Urban
Bidar
Chamarajanagar
Chikkaballapura
Chikkamagaluru
Chitradurga
Dakshina Kannada
Davanagere
Dharwad
Gadag
Hassan
Haveri
Kalaburagi
Kodagu
Kolar
Koppal
Mandya
Mysuru
Raichur
Shivamogga
Tumakuru
Udupi
Uttara Kannada
Vijayanagara
Vijayapura
Yadgir

Rows per year (last 10):
year
2013-2014    1615
2014-2015    1581
2015-2016    1609
2016-2017    1548
2017-2018    1529
2018-2019    1676
2019-2020    1967
2020-2021    2190
2021-2022    2044
2022-2023    2112
dtype: int64

Share missing:
area          0.000
production    0.004
yield         0.000
dtype: float64

Yield units: {'Tonnes/Hectare': 39788}
Production units: {'Tonnes': 39788}

Yield summary (tonnes/hectare):
count    39788.000
mean       125.194
std        980.178
min          0.000
25%          0.490
50%          1.031
75%          2.999
max      17590.295
Name: yield, dtype: float64
Rows with zero yield: 304

Coastal districts found: ['Dakshina Kannada', 'Ud

In [4]:
import os
import numpy as np

CROPS = ['Rice', 'Maize', 'Ragi', 'Jowar', 'Bajra',
         'Urad', 'Moong(Green Gram)', 'Cowpea(Lobia)', 'Horse-Gram', 'Arhar/Tur', 'Gram',
         'Groundnut', 'Sunflower', 'Sesamum']
SEASONS = ['Kharif', 'Rabi', 'Summer']

d = ka.copy()
for col in ['district_name', 'season', 'crop_name', 'year']:
    d[col] = d[col].astype(str).str.strip()          # remove stray spaces

missing = [c for c in CROPS if c not in d['crop_name'].unique()]
print("Crops not found (check spelling):", missing)

log = [("Start: Karnataka rows", len(d))]

# Rule 1: real seasons only (drops 'Total' double-counts and 'Whole Year' crops)
d = d[d['season'].isin(SEASONS)]
log.append(("Rule 1: keep Kharif/Rabi/Summer", len(d)))

# Rule 2: our 14 crops only
d = d[d['crop_name'].isin(CROPS)]
log.append(("Rule 2: keep our 14 crops", len(d)))

# Rule 3: need real area and production
d = d[(d['area'] > 0) & (d['production'] > 0)]
log.append(("Rule 3: drop zero/missing area or production", len(d)))

# Check: does the reported yield match production / area?
calc = d['production'] / d['area']
off = ((calc - d['yield']).abs() > 0.01 * calc).mean()
print(f"Rows where reported yield differs from production/area by >1%: {off:.1%}")

# Rule 4: year '2015-2016' -> 2015 (start of the agricultural year)
d['year_start'] = d['year'].str[:4].astype(int)

# Rule 5: merge Vijayanagara back into Ballari, then recompute yield ourselves
d['district_name'] = d['district_name'].replace({'Vijayanagara': 'Ballari'})
d = (d.groupby(['year_start', 'district_name', 'season', 'crop_name'], as_index=False)
       [['area', 'production']].sum())
d['yield_t_ha'] = d['production'] / d['area']
log.append(("Rule 5: merge Vijayanagara -> Ballari", len(d)))

# Rule 6: remove extreme yields, judged separately for each crop
q1 = d.groupby('crop_name')['yield_t_ha'].transform(lambda s: s.quantile(0.25))
q3 = d.groupby('crop_name')['yield_t_ha'].transform(lambda s: s.quantile(0.75))
too_high = d['yield_t_ha'] > q3 + 3 * (q3 - q1)
print("\nExamples of rows removed as extreme:")
print(d[too_high].sort_values('yield_t_ha', ascending=False).head(8).round(2))
d = d[~too_high]
log.append(("Rule 6: drop extreme yields (per crop)", len(d)))

# Cleaning log: before -> after
print("\nCLEANING LOG")
for step, n in log:
    print(f"{step:<45} {n:>7,}")

# Sanity checks on the result
print("\nYield per crop (t/ha):")
print(d.groupby('crop_name')['yield_t_ha'].describe()[['count', 'min', '50%', 'max']].round(2))
print("\nRows per year:")
print(d.groupby('year_start').size().to_string())

# Save the silver table
os.makedirs('/content/drive/MyDrive/crop_reco/silver', exist_ok=True)
out = '/content/drive/MyDrive/crop_reco/silver/yield_clean.csv'
d.to_csv(out, index=False)
print("\nSaved:", out, "| rows:", len(d))

Crops not found (check spelling): []
Rows where reported yield differs from production/area by >1%: 0.2%

Examples of rows removed as extreme:
       year_start    district_name  season crop_name    area  production  \
11286        2016  Bengaluru South  Summer     Maize   133.0      1276.0   
12977        2018          Raichur  Kharif     Maize  4077.0     30974.0   
14088        2020  Bengaluru Urban  Kharif     Maize  1042.0      7895.0   
12220        2017           Kodagu  Kharif     Maize  3558.0     26743.0   
5439         2007  Bengaluru South  Kharif     Maize  1669.0     12220.0   
11366        2016  Chikkaballapura  Summer     Maize   369.0      2624.0   
16168        2022          Raichur  Kharif     Maize   660.0      4435.0   
13279        2019  Bengaluru Urban  Kharif     Maize  2073.0     13874.0   

       yield_t_ha  
11286        9.59  
12977        7.60  
14088        7.58  
12220        7.52  
5439         7.32  
11366        7.11  
16168        6.72  
13279       

In [5]:
import ee
ee.Initialize(project="crop-reco-me")    # already signed in earlier this session

gaul = (ee.FeatureCollection("FAO/GAUL/2025/level2")
          .filter(ee.Filter.eq("GAUL1_NAME", "Karnataka")))

map_names = sorted(gaul.aggregate_array("GAUL2_NAME").getInfo())
yield_names = sorted(d['district_name'].unique())

print("Districts on the map:", len(map_names))
print("\n".join(map_names))
print("\nIn yield data but NOT on the map:", [n for n in yield_names if n not in map_names])
print("On the map but NOT in yield data:", [n for n in map_names if n not in yield_names])

Districts on the map: 30
Bagalkot
Ballari
Belagavi
Bengaluru Rural
Bengaluru Urban
Bidar
Chamarajanagar
Chik Ballapur
Chikkamagaluru
Chitradurga
Dakshina  Kannada
Davangere
Dharwad
Gadag
Hassan
Haveri
Kalaburagi
Kodagu
Kolar
Koppal
Mandya
Mysuru
Raichur
Ramanagaram
Shivamogga
Tumakuru
Udupi
Uttara  Kannada
Vijayapura
Yadgir

In yield data but NOT on the map: ['Bagalkote', 'Bengaluru South', 'Chikkaballapura', 'Dakshina Kannada', 'Davanagere', 'Uttara Kannada']
On the map but NOT in yield data: ['Bagalkot', 'Chik Ballapur', 'Dakshina  Kannada', 'Davangere', 'Ramanagaram', 'Uttara  Kannada']


In [6]:
import ee, time
ee.Initialize(project="crop-reco-me")

YEARS = list(range(2000, 2023))
FOLDER = "crop_reco_gee"          # new Drive folder the files will appear in

# ---------- 1. Districts with names matching the yield data ----------
RENAME = {"Bagalkot": "Bagalkote", "Chik Ballapur": "Chikkaballapura",
          "Davangere": "Davanagere", "Ramanagaram": "Bengaluru South"}

def fix_name(f):
    raw = ee.String(f.get("GAUL2_NAME")).replace("\\s+", " ", "g").trim()   # collapse double spaces
    return f.set("district_name", ee.Dictionary(RENAME).get(raw, raw))

districts = (ee.FeatureCollection("FAO/GAUL/2025/level2")
               .filter(ee.Filter.eq("GAUL1_NAME", "Karnataka"))
               .map(fix_name)
               .select(["district_name"]))

names = set(districts.aggregate_array("district_name").getInfo())
print("Still unmatched (should be empty):", sorted(names ^ set(d['district_name'].unique())))

# ---------- 2. Cropland mask (ESA WorldCover 2020, class 40 = cropland) ----------
crop_mask = ee.ImageCollection("ESA/WorldCover/v100").first().select("Map").eq(40)

# ---------- 3. Static soil + terrain, averaged over cropland ----------
OLM = "OpenLandMap/SOL/"
def topsoil(asset, name):        # average of 0, 10 and 30 cm depths
    return ee.Image(OLM + asset).select(["b0", "b10", "b30"]).reduce(ee.Reducer.mean()).rename(name)

srtm = ee.Image("USGS/SRTMGL1_003")
static_img = ee.Image.cat([
    topsoil("SOL_PH-H2O_USDA-4C1A2A_M/v02", "ph_x10"),
    topsoil("SOL_ORGANIC-CARBON_USDA-6A1C_M/v02", "oc_x5gkg"),
    topsoil("SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02", "clay_pct"),
    topsoil("SOL_SAND-WFRACTION_USDA-3A1A1A_M/v02", "sand_pct"),
    srtm.rename("elev_m"),
    ee.Terrain.slope(srtm).rename("slope_deg"),
]).updateMask(crop_mask).addBands(crop_mask.rename("crop_frac"))

static_fc = static_img.reduceRegions(collection=districts, reducer=ee.Reducer.mean(),
                                     scale=250, tileScale=8)

# ---------- 4. Season weather for each year ----------
# (year offset, month) for start and end; end month is exclusive
WINDOWS = {"Kharif": ((0, 6), (0, 11)),    # Jun–Oct of year Y
           "Rabi":   ((0, 11), (1, 3)),    # Nov Y – Feb Y+1
           "Summer": ((1, 3), (1, 6))}     # Mar–May of Y+1

def season_image(year, season):
    (ys, ms), (ye, me) = WINDOWS[season]
    start = ee.Date.fromYMD(year + ys, ms, 1)
    end = ee.Date.fromYMD(year + ye, me, 1)
    rain_daily = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterDate(start, end).select("precipitation")
    era = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR").filterDate(start, end)
    return ee.Image.cat([
        rain_daily.sum().rename("rain_mm"),
        rain_daily.map(lambda i: i.gt(2.5)).sum().rename("rainy_days"),
        era.select("temperature_2m").mean().subtract(273.15).rename("tmean_c"),
        era.select("temperature_2m_max").mean().subtract(273.15).rename("tmax_c"),
        era.select("temperature_2m_min").mean().subtract(273.15).rename("tmin_c"),
    ])

parts = []
for y in YEARS:
    for s in WINDOWS:
        fc = season_image(y, s).reduceRegions(collection=districts, reducer=ee.Reducer.mean(),
                                              scale=5000, tileScale=4)
        parts.append(fc.map(lambda f, y=y, s=s: f.set({"year_start": y, "season": s})))
season_fc = ee.FeatureCollection(parts).flatten()

# ---------- 5. Export both tables to Google Drive ----------
def export(fc, name, cols):
    t = ee.batch.Export.table.toDrive(collection=fc, description=name, folder=FOLDER,
                                      fileNamePrefix=name, fileFormat="CSV", selectors=cols)
    t.start()
    return t

tasks = [
    export(static_fc, "district_static",
           ["district_name", "ph_x10", "oc_x5gkg", "clay_pct", "sand_pct", "elev_m", "slope_deg", "crop_frac"]),
    export(season_fc, "district_season_weather",
           ["district_name", "year_start", "season", "rain_mm", "rainy_days", "tmean_c", "tmax_c", "tmin_c"]),
]

# ---------- 6. Wait and report ----------
while True:
    states = [t.status()["state"] for t in tasks]
    print(time.strftime("%H:%M:%S"), states)
    if all(s in ("COMPLETED", "FAILED", "CANCELLED") for s in states):
        break
    time.sleep(30)

for t in tasks:
    st = t.status()
    print(st["description"], "->", st["state"], st.get("error_message", ""))

Still unmatched (should be empty): []
14:53:34 ['READY', 'READY']
14:54:05 ['COMPLETED', 'COMPLETED']
district_static -> COMPLETED 
district_season_weather -> COMPLETED 


In [7]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)    # refresh so Colab sees the new files

import glob, os
import pandas as pd

GEE = '/content/drive/MyDrive/crop_reco_gee'
static = pd.read_csv(glob.glob(f'{GEE}/district_static*.csv')[0])
weather = pd.read_csv(glob.glob(f'{GEE}/district_season_weather*.csv')[0])

# ---------- 1. Shape and gaps ----------
print("static:", static.shape, "| weather:", weather.shape)
print("Missing in static:", static.isna().sum()[lambda s: s > 0].to_dict())
print("Missing in weather:", weather.isna().sum()[lambda s: s > 0].to_dict())

# ---------- 2. Convert to real units ----------
static['ph'] = static['ph_x10'] / 10            # stored as pH x 10
static['oc_gkg'] = static['oc_x5gkg'] * 5       # stored in units of 5 g/kg
static = static.drop(columns=['ph_x10', 'oc_x5gkg'])

# ---------- 3. Sanity checks against known agronomy ----------
print("\nSoil by district (sorted by pH):")
print(static.sort_values('ph').round(2).to_string(index=False))

print("\nAverage weather by season, all of Karnataka:")
print(weather.groupby('season')[['rain_mm', 'rainy_days', 'tmean_c', 'tmax_c', 'tmin_c']].mean().round(1))

print("\nAverage weather by season, coast vs rest:")
weather['zone'] = weather['district_name'].isin(['Udupi', 'Dakshina Kannada', 'Uttara Kannada']).map(
    {True: 'coast', False: 'rest'})
print(weather.groupby(['zone', 'season'])[['rain_mm', 'tmean_c']].mean().round(1))
weather = weather.drop(columns='zone')

# ---------- 4. Join into the gold table ----------
y = pd.read_csv('/content/drive/MyDrive/crop_reco/silver/yield_clean.csv')
y = y[y['year_start'].between(2000, 2022)]

gold = (y.merge(weather, on=['district_name', 'year_start', 'season'], how='left')
          .merge(static, on='district_name', how='left'))

print("\nYield rows 2000-2022:", len(y), "| gold rows:", len(gold))
print("Rows missing weather:", gold['rain_mm'].isna().sum(),
      "| rows missing soil:", gold['ph'].isna().sum())

os.makedirs('/content/drive/MyDrive/crop_reco/gold', exist_ok=True)
gold.to_csv('/content/drive/MyDrive/crop_reco/gold/training_table.csv', index=False)

print("\nOne example row (Udupi, Rabi, Urad):")
ex = gold[(gold.district_name == 'Udupi') & (gold.season == 'Rabi') & (gold.crop_name == 'Urad')]
print(ex.head(1).T.to_string())

Mounted at /content/drive
static: (30, 8) | weather: (2070, 8)
Missing in static: {}
Missing in weather: {}

Soil by district (sorted by pH):
   district_name  clay_pct  sand_pct  elev_m  slope_deg  crop_frac   ph  oc_gkg
Dakshina Kannada     28.93     44.18   41.33       1.21       0.00 5.43   16.54
          Kodagu     34.44     39.60  873.81       2.02       0.03 5.51   15.53
           Udupi     28.88     42.02   18.10       0.94       0.04 5.52   15.51
  Uttara Kannada     29.77     44.84  489.09       1.35       0.06 5.87   10.23
      Shivamogga     29.93     46.17  617.70       1.43       0.22 6.00   10.14
 Bengaluru South     29.73     51.03  771.13       1.67       0.31 6.28    9.71
 Bengaluru Urban     30.45     50.79  875.83       1.24       0.37 6.33    9.17
          Hassan     30.38     49.24  894.05       1.43       0.47 6.41    9.54
 Bengaluru Rural     29.73     51.65  892.85       1.19       0.67 6.43    8.99
          Mandya     30.53     49.81  758.83       1.19   

In [8]:
%pip install -q lightgbm
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold

gold = pd.read_csv('/content/drive/MyDrive/crop_reco/gold/training_table.csv')

FEATURES = ['crop_name', 'season', 'rain_mm', 'rainy_days', 'tmean_c', 'tmax_c', 'tmin_c',
            'clay_pct', 'sand_pct', 'elev_m', 'slope_deg', 'ph', 'oc_gkg']
X = gold[FEATURES].copy()
for c in ['crop_name', 'season']:
    X[c] = X[c].astype('category')          # tell LightGBM these are labels, not numbers
y = gold['yield_t_ha']
w = gold['area']                              # bigger areas = more reliable yields

def make_model():
    return lgb.LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=31,
                             min_child_samples=30, subsample=0.8, subsample_freq=1,
                             colsample_bytree=0.8, random_state=42, verbose=-1)

def scores(true, pred):
    return {"MAE (t/ha)": mean_absolute_error(true, pred), "R2": r2_score(true, pred)}

# ================= TEST A: TIME (train 2000-2017, test 2018-2022) =================
tr = gold['year_start'] <= 2017
te = ~tr

crop_median = gold[tr].groupby('crop_name')['yield_t_ha'].median()
b0 = gold.loc[te, 'crop_name'].map(crop_median)

hist = gold[tr].groupby(['district_name', 'season', 'crop_name'])['yield_t_ha'].mean()
keys = zip(gold.loc[te, 'district_name'], gold.loc[te, 'season'], gold.loc[te, 'crop_name'])
b1 = pd.Series([hist.get(k, np.nan) for k in keys], index=gold.index[te]).fillna(b0)

model = make_model().fit(X[tr], y[tr], sample_weight=w[tr])
pred_A = pd.Series(model.predict(X[te]), index=gold.index[te])

print("TEST A - TIME  | train rows:", tr.sum(), "| test rows:", te.sum())
print(pd.DataFrame({"Baseline 0 (crop median)": scores(y[te], b0),
                    "Baseline 1 (district history)": scores(y[te], b1),
                    "Model (weather + soil)": scores(y[te], pred_A)}).T.round(3))

# Per-crop view: error as % of that crop's typical yield
per_crop = pd.DataFrame({
    "median_yield": crop_median,
    "MAE_hist": (y[te] - b1).abs().groupby(gold.loc[te, 'crop_name']).mean(),
    "MAE_model": (y[te] - pred_A).abs().groupby(gold.loc[te, 'crop_name']).mean(),
})
per_crop["hist_err_%"] = 100 * per_crop["MAE_hist"] / per_crop["median_yield"]
per_crop["model_err_%"] = 100 * per_crop["MAE_model"] / per_crop["median_yield"]
print("\nPer crop (Test A):")
print(per_crop.round(2).sort_values("model_err_%"))

# ================= TEST B: SPACE (whole districts hidden) =================
oof = pd.Series(np.nan, index=gold.index)
oof_b0 = pd.Series(np.nan, index=gold.index)
for fold, (i_tr, i_te) in enumerate(GroupKFold(n_splits=5).split(X, y, groups=gold['district_name'])):
    m = make_model().fit(X.iloc[i_tr], y.iloc[i_tr], sample_weight=w.iloc[i_tr])
    oof.iloc[i_te] = m.predict(X.iloc[i_te])
    med = gold.iloc[i_tr].groupby('crop_name')['yield_t_ha'].median()
    oof_b0.iloc[i_te] = gold.iloc[i_te]['crop_name'].map(med).values
    print(f"fold {fold}: hidden districts = {sorted(gold.iloc[i_te]['district_name'].unique())}")

print("\nTEST B - SPACE (unseen districts)")
print(pd.DataFrame({"Baseline 0 (crop median)": scores(y, oof_b0),
                    "Model (weather + soil)": scores(y, oof)}).T.round(3))

# What does the model rely on?
imp = pd.Series(model.booster_.feature_importance('gain'), index=FEATURES)
print("\nFeature importance (share of total gain, Test A model):")
print((100 * imp / imp.sum()).sort_values(ascending=False).round(1))

TEST A - TIME  | train rows: 11042 | test rows: 3875
                               MAE (t/ha)     R2
Baseline 0 (crop median)            0.328  0.803
Baseline 1 (district history)       0.292  0.849
Model (weather + soil)              0.304  0.841

Per crop (Test A):
                   median_yield  MAE_hist  MAE_model  hist_err_%  model_err_%
crop_name                                                                    
Rice                       2.64      0.40       0.44       15.09        16.58
Ragi                       1.59      0.21       0.27       13.45        17.03
Maize                      3.00      0.63       0.63       21.08        20.94
Gram                       0.53      0.15       0.12       28.21        23.38
Jowar                      1.10      0.37       0.35       33.44        31.80
Arhar/Tur                  0.51      0.17       0.17       33.47        34.10
Horse-Gram                 0.45      0.20       0.17       45.22        37.65
Groundnut                  0.

In [9]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

gold = pd.read_csv('/content/drive/MyDrive/crop_reco/gold/training_table.csv')
weather = pd.read_csv(glob.glob('/content/drive/MyDrive/crop_reco_gee/district_season_weather*.csv')[0])
KEY = ['district_name', 'season', 'crop_name']
WCOLS = ['rain_mm', 'rainy_days', 'tmean_c', 'tmax_c', 'tmin_c']
TRAIN_END = 2017

# ---------- 1. Rolling "normal" yield: mean of the previous 5 recorded seasons ----------
g = gold.sort_values('year_start').copy()
g['normal_yield'] = g.groupby(KEY)['yield_t_ha'].transform(
    lambda s: s.shift(1).rolling(5, min_periods=3).mean())
before = len(g)
g = g.dropna(subset=['normal_yield'])
print(f"Rows with at least 3 past seasons of history: {len(g):,} of {before:,}")

g['ratio'] = (g['yield_t_ha'] / g['normal_yield']).clip(0.2, 3.0)   # 1.0 = normal year

# ---------- 2. Weather anomalies vs normals from TRAINING years only ----------
wn = (weather[weather['year_start'] <= TRAIN_END]
        .groupby(['district_name', 'season'])[WCOLS].mean().add_suffix('_normal').reset_index())
g = g.merge(wn, on=['district_name', 'season'], how='left')
g['rain_anom'] = (g['rain_mm'] + 10) / (g['rain_mm_normal'] + 10)     # 1.0 = normal; +10 avoids /0 in dry seasons
g['rainy_days_anom'] = g['rainy_days'] - g['rainy_days_normal']
for t in ['tmean_c', 'tmax_c', 'tmin_c']:
    g[f'{t}_anom'] = g[t] - g[f'{t}_normal']

FEATS = ['crop_name', 'season', 'rain_anom', 'rainy_days_anom',
         'tmean_c_anom', 'tmax_c_anom', 'tmin_c_anom']
X = g[FEATS].copy()
for c in ['crop_name', 'season']:
    X[c] = X[c].astype('category')

tr = g['year_start'] <= TRAIN_END
te = ~tr

# ---------- 3. Train on the ratio ----------
m2 = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03, num_leaves=15,
                       min_child_samples=50, subsample=0.8, subsample_freq=1,
                       colsample_bytree=0.8, random_state=42, verbose=-1)
m2.fit(X[tr], g.loc[tr, 'ratio'], sample_weight=g.loc[tr, 'area'])

pred_hist = g.loc[te, 'normal_yield']                                  # history only
pred_v2 = g.loc[te, 'normal_yield'] * m2.predict(X[te])               # history x weather
true = g.loc[te, 'yield_t_ha']

def mae(p, mask=None):
    mask = slice(None) if mask is None else mask
    return round(mean_absolute_error(true[mask], p[mask]), 3)

unusual = (g.loc[te, 'ratio'] - 1).abs() > 0.3
print(f"\nTEST A (train <= {TRAIN_END}, test after) | test rows: {te.sum():,} | unusual years: {unusual.sum():,}")
print(pd.DataFrame({
    "All test rows": [mae(pred_hist), mae(pred_v2)],
    "Unusual years only": [mae(pred_hist, unusual), mae(pred_v2, unusual)],
}, index=["Rolling 5-yr history", "Model v2 (history x weather)"]))

# Per-crop: does weather help, and where?
pc = pd.DataFrame({
    "MAE_hist": (true - pred_hist).abs().groupby(g.loc[te, 'crop_name']).mean(),
    "MAE_v2": (true - pred_v2).abs().groupby(g.loc[te, 'crop_name']).mean(),
})
pc["improvement_%"] = 100 * (pc["MAE_hist"] - pc["MAE_v2"]) / pc["MAE_hist"]
print("\nPer crop (positive = weather helps):")
print(pc.round(3).sort_values("improvement_%", ascending=False))

imp = pd.Series(m2.booster_.feature_importance('gain'), index=FEATS)
print("\nFeature importance (% of gain):")
print((100 * imp / imp.sum()).sort_values(ascending=False).round(1))

# What the model learned: drought response per crop (kharif rain at 60% of normal)
probe = pd.DataFrame({'crop_name': sorted(g['crop_name'].unique())})
probe['season'] = 'Kharif'
probe['rain_anom'] = 0.6
for c in ['rainy_days_anom', 'tmean_c_anom', 'tmax_c_anom', 'tmin_c_anom']:
    probe[c] = 0.0
for c in ['crop_name', 'season']:
    probe[c] = pd.Categorical(probe[c], categories=X[c].cat.categories)
probe['predicted_ratio_if_kharif_rain_60%'] = m2.predict(probe[FEATS])
print("\nPredicted yield vs normal when kharif rain is 40% below normal:")
print(probe[['crop_name', 'predicted_ratio_if_kharif_rain_60%']].round(2).to_string(index=False))

Rows with at least 3 past seasons of history: 12,035 of 14,917

TEST A (train <= 2017, test after) | test rows: 3,498 | unusual years: 984
                              All test rows  Unusual years only
Rolling 5-yr history                  0.222               0.392
Model v2 (history x weather)          0.280               0.314

Per crop (positive = weather helps):
                   MAE_hist  MAE_v2  improvement_%
crop_name                                         
Sunflower             0.217   0.211          2.951
Cowpea(Lobia)         0.118   0.116          2.090
Horse-Gram            0.162   0.169         -4.277
Groundnut             0.258   0.271         -5.124
Moong(Green Gram)     0.077   0.085         -9.800
Gram                  0.132   0.150        -13.902
Sesamum               0.223   0.258        -15.854
Jowar                 0.264   0.315        -19.154
Maize                 0.457   0.551        -20.458
Rice                  0.311   0.410        -31.625
Bajra              

In [10]:
from sklearn.metrics import mean_absolute_error

def fit_ratio_model(mask):
    m = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03, num_leaves=15,
                          min_child_samples=50, subsample=0.8, subsample_freq=1,
                          colsample_bytree=0.8, random_state=42, verbose=-1)
    return m.fit(X[mask], g.loc[mask, 'ratio'], sample_weight=g.loc[mask, 'area'])

# ---------- 1. Choose alpha on a validation period INSIDE the training years ----------
inner_tr = g['year_start'] <= 2013
val = g['year_start'].between(2014, 2017)
m_inner = fit_ratio_model(inner_tr)
r_val = m_inner.predict(X[val])

alphas = np.round(np.linspace(0, 1, 11), 1)
val_mae = [mean_absolute_error(g.loc[val, 'yield_t_ha'],
                               g.loc[val, 'normal_yield'] * (1 + a * (r_val - 1)))
           for a in alphas]
best_alpha = alphas[int(np.argmin(val_mae))]
print("Validation (2014-2017) MAE by alpha:")
print(pd.Series(val_mae, index=alphas).round(4).to_string())
print("Best alpha:", best_alpha)

# ---------- 2. Test once on 2018-2022 with the chosen alpha ----------
r_te = m2.predict(X[te])                       # model trained on <= 2017 (from the previous cell)
true = g.loc[te, 'yield_t_ha']
hist = g.loc[te, 'normal_yield']
preds = {
    "Rolling 5-yr history": hist,
    "Model v2 (alpha = 1)": hist * r_te,
    f"Blend (alpha = {best_alpha})": hist * (1 + best_alpha * (r_te - 1)),
}
unusual = (g.loc[te, 'ratio'] - 1).abs() > 0.3
print("\nTEST 2018-2022:")
print(pd.DataFrame({
    "All rows": {k: mean_absolute_error(true, p) for k, p in preds.items()},
    "Unusual years": {k: mean_absolute_error(true[unusual], p[unusual]) for k, p in preds.items()},
}).round(3))

# ---------- 3. Apply the pre-agreed decision rule ----------
blend = preds[f"Blend (alpha = {best_alpha})"]
keep = (best_alpha > 0
        and mean_absolute_error(true, blend) < mean_absolute_error(true, hist)
        and mean_absolute_error(true[unusual], blend[unusual]) < mean_absolute_error(true[unusual], hist[unusual]))
print("\nDECISION:", "KEEP weather model as risk component" if keep
      else "DROP weather model from final ranking (document as experiment)")

Validation (2014-2017) MAE by alpha:
0.0    0.3041
0.1    0.2994
0.2    0.2960
0.3    0.2942
0.4    0.2937
0.5    0.2943
0.6    0.2962
0.7    0.2993
0.8    0.3039
0.9    0.3097
1.0    0.3165
Best alpha: 0.4

TEST 2018-2022:
                      All rows  Unusual years
Rolling 5-yr history     0.222          0.392
Model v2 (alpha = 1)     0.280          0.314
Blend (alpha = 0.4)      0.222          0.356

DECISION: DROP weather model from final ranking (document as experiment)


In [11]:
gold = pd.read_csv('/content/drive/MyDrive/crop_reco/gold/training_table.csv')
KEY = ['district_name', 'season', 'crop_name']
gs = gold.sort_values('year_start')

# 1. Expected yield = mean of the last 5 recorded seasons
expected = (gs.groupby(KEY).tail(5)
              .groupby(KEY).agg(expected_yield=('yield_t_ha', 'mean'),
                                recent_area_ha=('area', 'mean'),
                                last_year=('year_start', 'max')))
n_seasons = gold.groupby(KEY).size().rename('n_seasons')

# 2. Risk = 20th percentile of (yield / its prior 5-yr normal) over the last 10 seasons
risk = (g.sort_values('year_start').groupby(KEY).tail(10)
          .groupby(KEY)['ratio'].agg(bad_year_ratio=lambda s: s.quantile(0.2),
                                     n_ratio='count'))

table = expected.join(n_seasons).join(risk)
table['bad_year_yield'] = table['expected_yield'] * table['bad_year_ratio']

# 3. Evidence label: how much should we trust this row?
table['evidence'] = np.select(
    [(table['n_seasons'] >= 10) & (table['last_year'] >= 2018) & (table['n_ratio'] >= 5),
     (table['n_seasons'] >= 5)],
    ['good', 'limited'], default='weak')
table = table.reset_index()

print("Rows:", len(table))
print("Evidence counts:", table['evidence'].value_counts().to_dict())

cols = ['crop_name', 'expected_yield', 'bad_year_yield', 'bad_year_ratio',
        'recent_area_ha', 'n_seasons', 'last_year', 'evidence']
for season in ['Kharif', 'Rabi', 'Summer']:
    t = table[(table.district_name == 'Udupi') & (table.season == season)]
    print(f"\n=== UDUPI - {season.upper()} ===")
    print(t[cols].sort_values('recent_area_ha', ascending=False).round(2).to_string(index=False))

table.to_csv('/content/drive/MyDrive/crop_reco/gold/yield_expectations.csv', index=False)
print("\nSaved: gold/yield_expectations.csv")

Rows: 1032
Evidence counts: {'good': 692, 'weak': 217, 'limited': 123}

=== UDUPI - KHARIF ===
        crop_name  expected_yield  bad_year_yield  bad_year_ratio  recent_area_ha  n_seasons  last_year evidence
             Rice            2.93            2.97            1.02        34563.60         23       2022     good
             Urad            0.52             NaN             NaN         1984.67          3       2020     weak
       Horse-Gram            0.52             NaN             NaN          928.00          1       2004     weak
        Groundnut            1.28             NaN             NaN          772.67          3       2018     weak
Moong(Green Gram)            0.16             NaN             NaN           44.00          1       2004     weak
          Sesamum            0.77            0.73            0.96           34.20         21       2022     good
            Jowar            1.51             NaN             NaN            4.67          3       2020     weak
 

In [12]:
def detrended_risk(s):
    s = s.sort_values('year_start').tail(10)              # last 10 recorded seasons
    yv = s['yield_t_ha'].to_numpy()
    flat = float((np.abs(np.diff(yv)) < 1e-3).mean()) if len(yv) > 1 else np.nan
    if len(s) < 6:
        return pd.Series({'bad_year_ratio': np.nan, 'n_risk': len(s), 'flat_share': flat})
    x = s['year_start'].to_numpy()
    slope, intercept = np.polyfit(x, yv, 1)              # straight trend line
    trend = np.clip(intercept + slope * x, 1e-6, None)
    ratio = yv / trend                                     # 1.0 = on trend; 0.7 = 30% below trend
    return pd.Series({'bad_year_ratio': np.quantile(ratio, 0.2), 'n_risk': len(s), 'flat_share': flat})

risk2 = gold.groupby(KEY)[['year_start', 'yield_t_ha']].apply(detrended_risk).reset_index()

table = table.drop(columns=['bad_year_ratio', 'bad_year_yield', 'n_ratio', 'evidence'], errors='ignore')
table = table.merge(risk2, on=KEY, how='left')
table['bad_year_yield'] = table['expected_yield'] * table['bad_year_ratio']
table['evidence'] = np.select(
    [(table['n_seasons'] >= 10) & (table['last_year'] >= 2018) & (table['n_risk'] >= 6),
     (table['n_seasons'] >= 5)],
    ['good', 'limited'], default='weak')

# Smoothed-data check: series where many consecutive years repeat the exact same yield
print("Share of series with >30% exact repeats:",
      round((table['flat_share'] > 0.3).mean(), 3))
print(table.sort_values('flat_share', ascending=False)
           [KEY + ['flat_share', 'n_seasons']].head(8).round(2).to_string(index=False))

print("\nBad-year ratio across all 'good' rows (should now be mostly below 1):")
print(table.loc[table.evidence == 'good', 'bad_year_ratio'].describe().round(2))

cols = ['crop_name', 'expected_yield', 'bad_year_yield', 'bad_year_ratio',
        'recent_area_ha', 'flat_share', 'evidence']
t = table[(table.district_name == 'Udupi') & (table.season == 'Rabi')]
print("\n=== UDUPI - RABI (revised risk) ===")
print(t[cols].sort_values('recent_area_ha', ascending=False).round(2).to_string(index=False))

# Look at the raw numbers behind two crops
for crop in ['Rice', 'Urad']:
    s = gold[(gold.district_name == 'Udupi') & (gold.season == 'Rabi') & (gold.crop_name == crop)]
    print(f"\nUdupi Rabi {crop}, last 10 yields:",
          s.sort_values('year_start').tail(10)[['year_start', 'yield_t_ha']].round(2).values.tolist())

table.to_csv('/content/drive/MyDrive/crop_reco/gold/yield_expectations.csv', index=False)
print("\nSaved (overwritten): gold/yield_expectations.csv")

Share of series with >30% exact repeats: 0.036
 district_name season crop_name  flat_share  n_seasons
Chikkamagaluru Summer      Urad         1.0          3
    Davanagere Summer     Bajra         1.0          5
Chamarajanagar Summer     Bajra         1.0          2
       Raichur   Rabi     Bajra         1.0          3
         Kolar Summer     Bajra         1.0          2
         Udupi Kharif Arhar/Tur         1.0          2
         Udupi Kharif     Bajra         1.0          2
        Mysuru   Rabi     Bajra         1.0          2

Bad-year ratio across all 'good' rows (should now be mostly below 1):
count    692.00
mean       0.84
std        0.10
min        0.27
25%        0.79
50%        0.86
75%        0.91
max        1.02
Name: bad_year_ratio, dtype: float64

=== UDUPI - RABI (revised risk) ===
        crop_name  expected_yield  bad_year_yield  bad_year_ratio  recent_area_ha  flat_share evidence
             Rice            2.59            2.50            0.96         4636.80 

In [13]:
import io, glob
import ee
import numpy as np
import pandas as pd
ee.Initialize(project="crop-reco-me")

BASE = '/content/drive/MyDrive/crop_reco'
GEE = '/content/drive/MyDrive/crop_reco_gee'

# ================= REFERENCE DATA =================
# FAO ECOCROP crop requirements (temp C, growing-season rain mm, pH), extracted from the Recocrop package
RULES = pd.read_csv(io.StringIO("""crop_name,ecocrop_name,tmin,topmn,topmx,tmax,rmin,ropmn,ropmx,rmax,phmin,phopmn,phopmx,phmax,texture
Rice,"Rice, paddy (Indica)",16,25,35,38,1000,1500,2000,4000,3.5,5.0,8.0,9.0,W
Maize,Maize,10,18,33,47,400,600,1200,1800,4.5,5.0,7.0,8.5,M O
Ragi,Finger millet,8,18,27,35,600,800,1100,4300,5.5,6.0,6.5,7.5,M
Jowar,Sorghum (med. altitude),10,24,35,40,300,500,1000,3000,5.0,6.0,7.0,8.5,M H
Bajra,Pearl millet,12,25,35,40,200,400,900,1700,4.5,5.0,6.5,8.3,L M
Urad,Urd bean,8,22,35,40,530,650,900,2430,4.5,5.5,6.5,7.5,M H
Moong(Green Gram),Mung bean,8,21,36,40,500,650,900,1250,4.3,5.5,6.2,8.3,M O
Cowpea(Lobia),Cowpea,10,20,35,40,400,600,1500,4100,4.3,5.5,7.5,8.8,M
Horse-Gram,Horse gram,15,20,30,35,300,600,1800,4300,5.0,5.5,7.0,7.5,L M
Arhar/Tur,Pigeon Pea,10,18,38,45,400,600,1500,4000,4.5,5.0,7.0,8.4,L M
Gram,Chick pea,7,15,29,35,300,600,1000,1800,4.7,6.0,8.5,9.5,M H
Groundnut,Groundnut,10,22,32,45,400,600,1500,4000,4.5,5.5,6.5,8.5,M
Sunflower,Sunflower,5,17,34,45,300,600,1000,1600,5.5,6.0,7.5,8.0,L M
Sesamum,Sesame seed,10,20,30,40,300,500,1000,1500,4.5,5.5,7.5,8.0,M
"""))

# Official MSP, Rs per quintal: Kharif 2026-27 (CCEA, May 2026); Gram = Rabi 2027-28 (CCEA, Sep 2026)
MSP = {'Rice': 2441, 'Jowar': 4023, 'Bajra': 2900, 'Ragi': 5205, 'Maize': 2410,
       'Arhar/Tur': 8450, 'Moong(Green Gram)': 8780, 'Urad': 8200, 'Groundnut': 7517,
       'Sunflower': 8343, 'Sesamum': 10346, 'Gram': 5958,
       'Cowpea(Lobia)': np.nan, 'Horse-Gram': np.nan}          # no MSP for these two
PADDY_TO_RICE = 0.67                                           # ASSUMPTION: yield data is milled-rice basis
price = pd.Series(MSP, dtype=float)
price['Rice'] = MSP['Rice'] / PADDY_TO_RICE

RESIDUAL_MOISTURE_CROPS = {'Urad', 'Moong(Green Gram)', 'Cowpea(Lobia)', 'Horse-Gram', 'Gram'}
TEX_NAME = {'H': 'heavy', 'M': 'medium', 'L': 'light'}
Q_PER_ACRE = 10 / 2.471                                        # t/ha -> quintal/acre

hist = pd.read_csv(f'{BASE}/gold/yield_expectations.csv')
weather = pd.read_csv(glob.glob(f'{GEE}/district_season_weather*.csv')[0])
normals = weather.groupby(['district_name', 'season'])[['rain_mm', 'tmean_c']].mean()

# ================= LOCATION LOOKUP (Earth Engine) =================
RENAME = {"Bagalkot": "Bagalkote", "Chik Ballapur": "Chikkaballapura",
          "Davangere": "Davanagere", "Ramanagaram": "Bengaluru South"}
def fix_name(f):
    raw = ee.String(f.get("GAUL2_NAME")).replace("\\s+", " ", "g").trim()
    return f.set("district_name", ee.Dictionary(RENAME).get(raw, raw))
districts = (ee.FeatureCollection("FAO/GAUL/2025/level2")
               .filter(ee.Filter.eq("GAUL1_NAME", "Karnataka")).map(fix_name))

OLM = "OpenLandMap/SOL/"
def topsoil(asset, name):
    return ee.Image(OLM + asset).select(["b0", "b10", "b30"]).reduce(ee.Reducer.mean()).rename(name)
soil_img = ee.Image.cat([topsoil("SOL_PH-H2O_USDA-4C1A2A_M/v02", "ph_x10"),
                         topsoil("SOL_CLAY-WFRACTION_USDA-3A1A1A_M/v02", "clay_pct"),
                         topsoil("SOL_SAND-WFRACTION_USDA-3A1A1A_M/v02", "sand_pct")])

def point_facts(lat, lon):
    pt = ee.Geometry.Point([lon, lat])
    hit = districts.filterBounds(pt)
    if hit.size().getInfo() == 0:
        raise ValueError("Location is outside Karnataka; the system only covers Karnataka.")
    district = ee.Feature(hit.first()).get("district_name").getInfo()
    s = soil_img.reduceRegion(ee.Reducer.first(), pt, 250).getInfo()
    if s.get("ph_x10") is None:
        raise ValueError("No soil-map value here (water or map gap). Try a point slightly inland.")
    return district, s["ph_x10"] / 10, s["clay_pct"], s["sand_pct"]

def texture_class(clay, sand):          # simplified: heavy = clay >= 40%, light = sand >= 65%
    return 'H' if clay >= 40 else ('L' if sand >= 65 else 'M')

# ================= RULES =================
LEVEL = {'ok': 0, 'caution': 1, 'excluded': 2}
def check(r, ph, tex, rain, tmean, season, irrigated):
    status, notes = 'ok', []
    def flag(level, msg):
        nonlocal status
        if LEVEL[level] > LEVEL[status]:
            status = level
        notes.append(msg)
    if ph < r.phmin or ph > r.phmax:
        flag('excluded', f"pH {ph:.1f} outside tolerable {r.phmin}-{r.phmax}")
    elif not (r.phopmn <= ph <= r.phopmx):
        flag('caution', f"pH {ph:.1f} outside optimum {r.phopmn}-{r.phopmx}")
    if tmean < r.tmin or tmean > r.tmax:
        flag('excluded', f"season temp {tmean:.0f}C outside tolerable {r.tmin}-{r.tmax}C")
    elif not (r.topmn <= tmean <= r.topmx):
        flag('caution', f"season temp {tmean:.0f}C outside optimum {r.topmn}-{r.topmx}C")
    if season == 'Kharif':
        if rain > r.rmax:
            flag('excluded', f"rain {rain:.0f} mm above max {r.rmax} mm (waterlogging)")
        elif rain > r.ropmx:
            flag('caution', f"rain {rain:.0f} mm above optimum {r.ropmx} mm")
        if not irrigated and rain < r.rmin:
            flag('excluded', f"rain {rain:.0f} mm below min {r.rmin} mm, no irrigation")
        elif not irrigated and rain < r.ropmn:
            flag('caution', f"rain {rain:.0f} mm below optimum {r.ropmn} mm")
    elif not irrigated:
        if r.crop_name in RESIDUAL_MOISTURE_CROPS:
            flag('caution', "no irrigation: relies on residual moisture after kharif rice")
        else:
            flag('excluded', f"needs irrigation in {season} (season rain ~{rain:.0f} mm)")
    if 'W' not in r.texture.split() and tex not in r.texture.split():
        flag('caution', f"{TEX_NAME[tex]} soil not among preferred textures ({r.texture})")
    return status, "; ".join(notes)

# ================= RECOMMEND =================
def fmt(x):
    return f"{x:,.0f}" if pd.notna(x) else "-"

def recommend(lat, lon, season, irrigated, soil_ph=None):
    district, map_ph, clay, sand = point_facts(lat, lon)
    ph = soil_ph if soil_ph is not None else map_ph
    tex = texture_class(clay, sand)
    rain, tmean = normals.loc[(district, season)]

    print("=" * 100)
    print(f"Location ({lat}, {lon}) -> {district} | Season: {season} | Irrigation: {'yes' if irrigated else 'no'}")
    print(f"Soil pH {ph:.1f} ({'soil test' if soil_ph is not None else 'soil map, 250 m estimate'}) | "
          f"clay {clay:.0f}% sand {sand:.0f}% -> {TEX_NAME[tex]} | normal season rain {rain:.0f} mm, temp {tmean:.1f}C")

    res = pd.DataFrame([dict(crop_name=r.crop_name, **dict(zip(['status', 'reasons'],
                        check(r, ph, tex, rain, tmean, season, irrigated))))
                        for r in RULES.itertuples(index=False)])
    h = hist[(hist.district_name == district) & (hist.season == season)]
    res = res.merge(h[['crop_name', 'expected_yield', 'bad_year_yield', 'recent_area_ha', 'evidence']],
                    on='crop_name', how='left')
    res['evidence'] = res['evidence'].fillna('no local record')
    res['price_rs_q'] = res['crop_name'].map(price)
    res['income_acre'] = res['expected_yield'] * Q_PER_ACRE * res['price_rs_q']
    res['bad_year_income_acre'] = res['bad_year_yield'] * Q_PER_ACRE * res['price_rs_q']

    usable = (res.status != 'excluded') & res.evidence.isin(['good', 'limited'])
    ranked = res[usable & res.price_rs_q.notna()].sort_values('income_acre', ascending=False)

    print("\nRECOMMENDED (ranked by expected gross income per acre at MSP; costs not included):")
    if ranked.empty:
        print("  none")
    for i, r in enumerate(ranked.itertuples(), 1):
        print(f"  {i}. {r.crop_name:<18} {r.status:<8} yield {r.expected_yield:.2f} t/ha | "
              f"income Rs {fmt(r.income_acre)}/acre (bad year Rs {fmt(r.bad_year_income_acre)}) | "
              f"evidence: {r.evidence}" + (f" | {r.reasons}" if r.reasons else ""))

    nop = res[usable & res.price_rs_q.isna()]
    if len(nop):
        print("\nALSO POSSIBLE, but no official price (check the local mandi):")
        for r in nop.itertuples():
            print(f"  - {r.crop_name}: yield {r.expected_yield:.2f} t/ha, evidence {r.evidence}"
                  + (f" | {r.reasons}" if r.reasons else ""))

    thin = res[(res.status != 'excluded') & ~res.evidence.isin(['good', 'limited'])]
    if len(thin):
        print("\nNOT ENOUGH LOCAL DATA to recommend:", ", ".join(thin.crop_name))

    bad = res[res.status == 'excluded']
    if len(bad):
        print("\nNOT SUITABLE HERE:")
        for r in bad.itertuples():
            print(f"  - {r.crop_name}: {r.reasons}")
    return res

# ================= SHANKAR: three scenarios =================
r1 = recommend(13.42, 74.76, 'Rabi', irrigated=True)                  # borewell, soil from map
r2 = recommend(13.42, 74.76, 'Rabi', irrigated=False)                 # no irrigation
r3 = recommend(13.42, 74.76, 'Rabi', irrigated=True, soil_ph=6.3)     # borewell + soil test pH 6.3

Location (13.42, 74.76) -> Udupi | Season: Rabi | Irrigation: yes
Soil pH 4.9 (soil map, 250 m estimate) | clay 28% sand 45% -> medium | normal season rain 92 mm, temp 25.7C

RECOMMENDED (ranked by expected gross income per acre at MSP; costs not included):
  1. Groundnut          caution  yield 2.12 t/ha | income Rs 64,593/acre (bad year Rs 60,333) | evidence: good | pH 4.9 outside optimum 5.5-6.5
  2. Rice               caution  yield 2.59 t/ha | income Rs 38,222/acre (bad year Rs 36,841) | evidence: good | pH 4.9 outside optimum 5.0-8.0
  3. Maize              caution  yield 3.35 t/ha | income Rs 32,663/acre (bad year Rs 32,391) | evidence: good | pH 4.9 outside optimum 5.0-7.0
  4. Urad               caution  yield 0.54 t/ha | income Rs 17,852/acre (bad year Rs 16,614) | evidence: good | pH 4.9 outside optimum 5.5-6.5
  5. Moong(Green Gram)  caution  yield 0.27 t/ha | income Rs 9,748/acre (bad year Rs 8,220) | evidence: good | pH 4.9 outside optimum 5.5-6.2

ALSO POSSIBLE, but no o

In [14]:
from google.colab import drive, files
drive.mount('/content/drive')

import sys, ee
ee.Authenticate()
sys.path.insert(0, '/content/drive/MyDrive/crop_reco/code')

import importlib, crop_engine
importlib.reload(crop_engine)                 # picks up the file even if an older copy was imported
eng = crop_engine.Engine()                    # loads data, Earth Engine, soil model and CLIP (CLIP: ~1 min first time)

# Test 1: same as before, no photo (should match your earlier scenario 1)
eng.print_report(eng.recommend(13.42, 74.76, 'Rabi', irrigated=True))

# Test 2: photos. Pick 2 images from your computer: one soil photo (any), and one selfie / non-soil photo
uploaded = files.upload()
for name in uploaded:
    print(f"\n### {name}")
    res = eng.recommend_from_photo(name, 'Rabi', irrigated=True)
    if res['status'] == 'need_location':      # photo has no GPS -> use Shankar's point for the test
        print("(no GPS in photo, using 13.42, 74.76 for this test)")
        res = eng.recommend_from_photo(name, 'Rabi', irrigated=True, lat=13.42, lon=74.76)
    eng.print_report(res)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Location (13.42, 74.76) -> Udupi | Season: Rabi | Irrigation: yes | Previous crop: not given
Soil pH 4.9 (soil map (250 m estimate)) | texture medium (soil map (250 m estimate); map says medium) | normal season rain 92 mm, temp 25.7C

RECOMMENDED (ranked by expected gross income per acre at MSP; cultivation costs NOT included):
  1. Groundnut          caution  yield 2.12 t/ha | income Rs 64,593/acre (bad year Rs 60,333/acre) | evidence good, ~1,720 ha in district | pH 4.9 outside optimum 5.5-6.5
  2. Rice               caution  yield 2.59 t/ha | income Rs 38,222/acre (bad year Rs 36,841/acre) | evidence good, ~4,637 ha in district | pH 4.9 outside optimum 5.0-8.0
  3. Maize              caution  yield 3.35 t/ha | income Rs 32,663/acre (bad year Rs 32,391/acre) | evidence good, ~33 ha in district | few farmers grow this here | pH 4.9 outside optimum 5.0-7.0
  4. Urad               caution  yield 0.54 t/ha | income Rs 17,852/acre (bad year Rs 16,614/acre) | evidence good, ~2,458 ha in di

Saving RUID46645305a33b43a09c1b4ea6bed399a9.jpg to RUID46645305a33b43a09c1b4ea6bed399a9.jpg

### RUID46645305a33b43a09c1b4ea6bed399a9.jpg
(no GPS in photo, using 13.42, 74.76 for this test)
Location (13.42, 74.76) -> Udupi [entered manually] | Season: Rabi | Irrigation: yes | Previous crop: not given
Photo: soil type Alluvial (confidence 100%) | probabilities {'Alluvial': 1.0, 'Black': 0.0, 'Clay': 0.0, 'Red/Laterite': 0.0}
Soil pH 4.9 (soil map (250 m estimate)) | texture medium (photo: Alluvial (100% confident); map says medium) | normal season rain 92 mm, temp 25.7C

RECOMMENDED (ranked by expected gross income per acre at MSP; cultivation costs NOT included):
  1. Groundnut          caution  yield 2.12 t/ha | income Rs 64,593/acre (bad year Rs 60,333/acre) | evidence good, ~1,720 ha in district | pH 4.9 outside optimum 5.5-6.5
  2. Rice               caution  yield 2.59 t/ha | income Rs 38,222/acre (bad year Rs 36,841/acre) | evidence good, ~4,637 ha in district | pH 4.9 outside op

In [15]:
%pip install -q gradio
from google.colab import drive
drive.mount('/content/drive')

import sys, importlib, ee
ee.Authenticate()
sys.path.insert(0, '/content/drive/MyDrive/crop_reco/code')
import crop_engine, crop_app
importlib.reload(crop_engine); importlib.reload(crop_app)

eng = crop_engine.Engine()          # ~1 min: data, Earth Engine, soil model, CLIP
demo = crop_app.build_app(eng)
demo.launch(share=True, debug=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3f43418c04dd791d38.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3f43418c04dd791d38.gradio.live
